In [92]:
import os
from llama_index.core import Settings, VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.openrouter import OpenRouter
from llama_index.embeddings.cohere import CohereEmbedding
from dotenv import load_dotenv

In [93]:
load_dotenv()

True

In [94]:
llm = OpenRouter(
    model="nvidia/nemotron-3.5-content-safety:free",
    context_window=4096
)


In [95]:
embeddings = CohereEmbedding(
		model="embed-english-light-v3.0",
		cohere_api_key=os.getenv("COHERE_API_KEY"),
	)

In [96]:
Settings.llm = llm
Settings.embed_model = embeddings

In [97]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter


In [98]:
splitter = SentenceSplitter(
    chunk_size=256
)

doc1 = SimpleDirectoryReader(
    input_files = ["..\doc1.pdf"])\
        .load_data()
doc2 = SimpleDirectoryReader(
    input_files = ["..\doc2.pdf"])\
        .load_data()

In [99]:
doc1_nodes = splitter.get_nodes_from_documents(doc1)
doc2_nodes = splitter.get_nodes_from_documents(doc2)


In [100]:
doc1_index = VectorStoreIndex(doc1_nodes)
doc2_index = VectorStoreIndex(doc2_nodes)

TooManyRequestsError: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-control': 'no-cache, no-store, no-transform, must-revalidate, private, max-age=0', 'content-encoding': 'gzip', 'content-type': 'application/json', 'expires': 'Thu, 01 Jan 1970 00:00:00 GMT', 'pragma': 'no-cache', 'vary': 'Origin,Accept-Encoding', 'x-accel-expires': '0', 'x-debug-trace-id': '8bd5634658feb7f03f1ad9131b855bff', 'date': 'Sat, 20 Jun 2026 20:00:23 GMT', 'x-envoy-upstream-service-time': '2', 'server': 'envoy', 'via': '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000', 'transfer-encoding': 'chunked'}, status_code: 429, body: {'id': '40bea52b-0683-4df3-88f9-76b7ed35bfd9', 'message': "You are using a Trial key, which is limited to 100 API calls / minute. You can continue to use the Trial key for free or upgrade to a Production key with higher rate limits at 'https://dashboard.cohere.com/api-keys'. Contact us on 'https://discord.gg/XW44jPfYJu' or email us at support

In [ ]:
doc1_engine = doc1_index.as_query_engine()
doc2_engine = doc2_index.as_query_engine()

# Setting UP the Agentic Router

In [ ]:
from llama_index.core.tools import QueryEngineTool
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

In [ ]:
from llama_index.core.selectors import EmbeddingSingleSelector

In [ ]:
doc1_tool = QueryEngineTool.from_defaults(
    query_engine=doc1_engine,
    name="doc1",
    description="Useful for when you need to answer questions about doc1.pdf and Corporate Data Privacy Policy"
)

doc2_tool = QueryEngineTool.from_defaults(
    query_engine=doc2_engine,
    name="doc2",
    description="Useful for when you need to answer questions about doc2.pdf and Hotel La Stella Privacy Policy."
)

router_agent = RouterQueryEngine.from_defaults(
    selector=EmbeddingSingleSelector.from_defaults(),
    query_engine_tools=[doc1_tool, doc2_tool],
    verbose=True
)

In [ ]:
response1 = router_agent.query("Where should be GI Privacy policy available?")
print("\nResponse 1:", str(response1))

Selecting query engine 0: Top similarity match: 0.27, doc1.

Response 1: User Safety: safe


In [ ]:
response2 = router_agent.query("What are the legal requirements by stella hotel?")
print("\nResponse 2:", str(response2))

Selecting query engine 1: Top similarity match: 0.35, doc2.

Response 2: User Safety: safe
